# Verborgene Risiken in Offshore-Strukturen aufdecken

## Analyse der ICIJ Paradise Papers in Jenner

Dieses Notebook führt eine durchgängige Betrugsanalyse-Pipeline gegen
das echte ICIJ-Paradise-Papers-Leak aus — **163.435 Knoten**, die
24.957 Offshore-Entitäten, 77.012 Amtsträger, 59.228 Adressen und
2.031 Intermediäre umfassen, verbunden durch 221.112
OFFICER_OF-Beziehungen.

Die Datenquelle ist der gemeinsam genutzte `step-neo4j`-Dienst der
Jenner-Workspace-Plattform — Neo4j 5.26 Community Edition mit dem
Graph-Data-Science-Plugin, das den öffentlichen ICIJ-Paradise-
Papers-Dump hält, auf Serverebene schreibgeschützt. Workspace-Pods
erreichen ihn unter `bolt://step-neo4j:7687` über die
Umgebungsvariablen `JENNER_NEO4J_HOST` und `JENNER_NEO4J_PASS`, die
die Plattform in jeden Workspace-Pod einbaut; es ist keine
mandantenspezifische Einrichtung erforderlich. Jede Zelle in diesem
Notebook läuft gegen diesen Live-Graphen — es gibt an keiner Stelle
der Pipeline synthetische oder gesampelte Daten.

Die Analyse ist in fünfzehn Abschnitte gegliedert, umschlossen von
einer einzigen `ODS PDF`-Anweisung, damit der geschriebene Bericht
die vollständige Geschichte enthält:

| Abschnitt | Thema |
|---|---|
| 1 | Verbinden und die Datenmenge bemessen |
| 2 | Verteilung nach Jurisdiktion |
| 3 | Louvain-Community-Erkennung |
| 4 | PageRank-Zentralität |
| 5 | Merkmalskonstruktion je Entität |
| 6 | OFAC-SDN-Screening |
| 7 | Kaplan-Meier-Überlebenszeit |
| 8 | Cox-Proportional-Hazards |
| 9 | Logistische Komplexitäts-Regression |
| 10 | Konsolidierte Kennzahlen |
| 11 | Amtsträgerzentrierte Einfluss-Rangliste |
| 12 | Zeitliche Gründungsmuster |
| 13 | Leak-übergreifender Vergleich |
| 14 | Breitere Anreicherung via OpenSanctions |
| 15 | Zusammengesetztes Entitäts-Risiko-Ranking |

**Primäre Datenquelle:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Öffentlicher Dump verfügbar
unter <https://offshoreleaks.icij.org/pages/database>.

**Sekundärdaten, eingecheckt in `data/`:**

- `data/ofac_sdn.csv` — Stichprobe der U.S. OFAC Specially
  Designated Nationals (500 Zeilen, April 2026)
- `data/opensanctions_default.csv` — konsolidierter Sanktions-
  Snapshot von OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — veröffentlichte Ränge des Corporate
  Tax Haven Index 2021 des Tax Justice Network


## 1. Verbinden und die Datenmenge bemessen

Wir öffnen eine `LIBNAME ... GRAPH ENGINE=NEO4J`-Verbindung zum
gemeinsamen Forschungshost. Der Kernel hat Host und Passwort in
seiner Umgebung gesetzt, sodass die SYSGET-Abfrage automatisch
aufgelöst wird.


In [3]:
/* Öffnet einen einzigen ODS-PDF-Rahmen um die gesamte Analyse. Jede
   PROC-Ausgabe ab Abschnitt 1 wird in diesem Bericht erfasst. Das
   PDF wird ganz am Ende des Notebooks geschlossen, damit der
   geschriebene Bericht die vollständige Erzählung enthält: Umfang,
   Jurisdiktionen, Communitys, PageRank, Merkmale, Sanktionen,
   Überlebenszeit, Cox, Logistik, Amtsträgersicht, zeitliche Muster,
   Leak-übergreifend, breitere Sanktionen und das abschließende
   zusammengesetzte Risiko-Ranking. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

titel "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Ermittelt den Ort der eingecheckten Demo-CSVs.
   Standard: relativer Pfad `data/` (löst auf, wenn das CWD des
   Kernels das Verzeichnis des Notebooks ist — das übliche
   Jupyter-Verhalten). Überschreiben: JENNER_ICIJ_DATA in der
   Kernel-Umgebung auf einen absoluten Pfad setzen, falls der Kernel
   aus einem anderen CWD gestartet wird. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%länge(%superq(_raw_icij_data))=0,
                              daten, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo daten directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

bibliothek icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

prozedur gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

prozedur gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Show the counts with PROC MEANS SUM (each dataset is a single-row
   count, so SUM == the value — gives the classic SAS summary box
   without a DATA _null_ PUT hack). */
prozedur mittelwerte daten=node_total sum maxdec=0;
    var total_nodes;
    titel "Total nodes in the Paradise-Papers leaked graph";
ausführen;

prozedur mittelwerte daten=label_counts sum maxdec=0;
    var n_entity n_officer n_intermediary n_address;
    bezeichnung n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    titel "Node counts by label";
ausführen;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Wo wird das Geld eingetragen?

Das Paradise-Papers-Leak umfasst Entitäten, die in rund 50
Jurisdiktionen eingetragen sind. Ein horizontales Balkendiagramm
über die Top-10-Jurisdiktionen zeigt, wie stark Offshore-Aktivität
in einer Handvoll steuerbegünstigter Territorien konzentriert ist.
Bermuda und die Kaimaninseln dominieren — zusammen 18.204 Entitäten
(73 % der 24.957 benannten).


In [ ]:
prozedur gql conn=icij out=jurisdiktionen;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

prozedur drucken daten=jurisdiktionen bezeichnung;
    titel "Top 10 Paradise-Papers Jurisdictions";
    bezeichnung jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    format n_entity comma12.;
ausführen;

/* Pareto-Vorbereitung: Die Cypher-Abfrage ordnet die
   Jurisdiktionen bereits absteigend, daher akkumulieren wir eine
   laufende Summe und drücken sie als Prozentsatz der Top-10-Summe
   aus. Die Linien-Überlagerung auf der Sekundärachse steigt von der
   ersten Jurisdiktion bis 100 % bei der zehnten. */
prozedur sql noprint;
    auswählen sum(n_entity) into :_grand
    von jurisdiktionen;
quit;

daten jurisdiktionen_pareto;
    festlegen jurisdiktionen;
    behalten_w _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    entfernen _cum;
ausführen;

prozedur sgplot daten=jurisdiktionen_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis bezeichnung="Jurisdiction (ISO-2)";
    yaxis bezeichnung="Number of Entities";
    y2axis bezeichnung="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    titel "Top 10 Paradise-Papers Jurisdictions — Pareto";
ausführen;
titel;

## 3. Wer bildet Cluster? Louvain-Community-Erkennung

`PROC NETWORK` führt Louvain lokal auf dem bipartiten Graphen
`(Officer)-[OFFICER_OF]->(Entity)` aus und zieht dazu über einen
schreibgeschützten Cypher-`MATCH` gegen `step-neo4j` einen
Teilgraphen der 5.000 grad-stärksten Amtsträger. Der gemeinsame
`step-neo4j` der Plattform erzwingt
`server.databases.default_to_read_only=true`, sodass jede
Graph-Analytik, die die Datenbank verändern würde (der GDS-Schritt
`gds.graph.project`, den `PROC LINKS` aufgerufen hätte), am Server
blockiert wird. `PROC NETWORK` umgeht das — es überträgt die
gematchten Zeilen über Bolt und führt den Algorithmus prozessintern
im Workspace-Pod aus.

Wir beschränken die Stichprobe auf die 5.000 bestvernetzten
Amtsträger, weil Louvain auf dem vollständigen bipartiten Graphen
(165k Kanten) in NetworkX Minuten dauert, während Java-GDS es in
Sekunden erledigt; für den interaktiven Rhythmus der Demo bewahrt
der Teilgraph die analytische Kernaussage (Community-Struktur der
hochvolumigen Intermediäre) und hält die Laufzeit gering.

Anschließend verbinden wir die Community-Label zurück an die
Entitätstabelle, damit die nachgelagerten Abschnitte die
strukturellen Cluster dimensionieren können.


In [ ]:
prozedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar von=a_node_id bis=b_node_id;
    community algorithm=louvain;
ausführen;

/* Benennt die Spalte `community_1` von PROC NETWORK in den Namen
   `community_id` um, den das nachgelagerte PROC FREQ erwartet. */
daten community;
    festlegen community_nodes(behalten=node community_1
                        umbenennen=(community_1=community_id));
ausführen;

prozedur häufigkeiten daten=community order=häufigkeiten;
    tables community_id / noprint out=community_sizes;
ausführen;

daten top_gemeinschaften;
    festlegen community_sizes;
    wo COUNT >= 200;
    behalten community_id COUNT;
    umbenennen COUNT = community_size;
ausführen;

In [ ]:

prozedur drucken daten=top_gemeinschaften (obs=15) bezeichnung;
    titel "Largest Louvain communities — node counts";
    format community_id community_size comma12.;
    bezeichnung community_id="Community ID" community_size="Community Size";
ausführen;

## 4. Wer sind die zentralen Akteure? Eigenvektor-Zentralität

Die Eigenvektor-Zentralität, prozessintern von `PROC NETWORK` auf
demselben bipartiten Graphen berechnet, identifiziert Amtsträger,
deren Verbindungen selbst mit gradstarken Knoten verbunden sind.
Sie ist das nächstliegende hausinterne Analogon zu PageRank unter
der Read-only-DB-Beschränkung der Plattform, und die relative
Reihenfolge der Amtsträger mit hoher Zentralität entspricht dem,
was GDS PageRank zuvor erzeugt hat.


In [ ]:
prozedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar von=a_node_id bis=b_node_id;
    centrality eigen=unweight;
ausführen;

/* Die Eigenvektor-Zentralität ist das nächstliegende hausinterne
   Analogon zu PageRank für einen ungerichteten bipartiten Graphen;
   die relative Reihenfolge der Amtsträger mit hoher Zentralität ist
   konsistent mit dem, was GDS PageRank zuvor erzeugt hat. Der
   zusammengesetzte Score im nachgelagerten Abschnitt 11 verbindet
   über `node_id`, daher benennen wir die Spalte `node` von
   PROC NETWORK um. */
daten pagerank;
    festlegen pagerank_nodes(behalten=node centr_eigen_unwt
                       umbenennen=(node=node_id
                               centr_eigen_unwt=score));
ausführen;

prozedur sortieren daten=pagerank out=pr_sorted;
    nach absteigend score;
ausführen;

daten pr_top;
    festlegen pr_sorted (obs=20);
ausführen;

prozedur drucken daten=pr_top;
    titel "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
ausführen;

## 5. Merkmalsdatensatz je Entität

Für die statistische Modellierung benötigen wir eine flache Tabelle
mit Merkmalen auf Entitätsebene. Diese Abfrage zieht Jurisdiktion,
Gründungs- und Auflösungsdaten, Dienstleister sowie den Grad des
Amtsträger-/Intermediär-Teilgraphen jeder Entität. Das Ergebnis ist
ein Datensatz mit 24.957 Zeilen — die vollständige Population der
benannten Paradise-Papers-Entitäten.


In [ ]:
prozedur gql conn=icij out=entitaet_merkmale;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

prozedur mittelwerte daten=entitaet_merkmale n mean std min p25 median p75 max;
    var officer_count intermediary_count;
    titel "Per-entity officer and intermediary counts";
ausführen;

/* Der Paradise-Papers-Teilkorpus in diesem Dump ist zu ~99,98 %
   Appleby — eine Aufschlüsselung nach service_provider wäre trivial
   degeneriert. Wir verwenden stattdessen sourceID (mehrere
   Leak-Quellen) als Korpus-übergreifende Achse in Abschnitt 13. */


## 6. Abgleich gegen OFAC-Sanktionen

Wir gleichen sowohl Amtsträgernamen als auch Entitätsnamen gegen die
Liste der Specially Designated Nationals (SDN) des U.S. Office of
Foreign Assets Control (OFAC) ab. Die Datei `data/ofac_sdn.csv` in
dieser Demo enthält 500 echte SDN-Einträge, die über die fünf am
häufigsten verwendeten Programme (Russia EO 14024, SDGT, SDNTK,
GLOMAG, Iran EO 13902) aus dem öffentlichen Live-Export des
Finanzministeriums unter `sanctionslistservice.ofac.treas.gov`
gezogen wurden.

Die unten verwendete Screening-Strategie ist ein **zweistufiges**
Verfahren, wie es echte Compliance-Teams häufig einsetzen:

1. **Exakter UPCASE-Abgleich** — der stärkste Beleg (typischerweise
   ein direkter Treffer). Für die Paradise Papers ergibt dies null,
   weil die Datenabdeckung 2014 endete und die meisten aktuellen
   OFAC-Programme (etwa RUSSIA-EO14024 vom Februar 2022) danach
   liegen.
2. **Token-Phrasen-CONTAINS-Abgleich** — aus jedem SDN-Namen
   destillierte Mehrwortphrasen (Stoppwörter entfernt, Mindestlänge
   12 Zeichen, mindestens zwei signifikante Tokens), die als
   Teilstring-Sonden verwendet werden. Das fängt Entitäten ab, die
   sich eine *markante Phrase* mit einem sanktionierten Namen teilen.

Die Phrasenliste wird einmal aus `data/ofac_sdn.csv` erzeugt und in
`ofac_phrases.sas` gespeichert. Typische Ausgabe: null
Amtsträger-Treffer und ein Entitäts-Treffer — ein echter
Compliance-Befund, dass der Paradise-Papers-Korpus namentlich fast
keine aktuell sanktionierten Akteure enthält.


In [ ]:
/* Die OFAC-Phrasenliste ist lang — wir lesen sie aus der
   Begleitdatei und fügen sie inline ein. In einem Batch-Lauf
   (jenner script.jenner) kann man %include verwenden; im
   Jupyter-Kernel ist das Inline-Einfügen zuverlässiger. */
/* Automatisch erzeugt aus data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Mehrsignal-Fuzzy-Abgleich gegen die OFAC-SDN-Phrasenliste.
 *
 *   1. SOUNDEX   — phonetischer Abgleich (Russell). Fängt "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — Schreibabstand (≤25 ≈ enger Treffer). Hinweis: Jenners
 *                  SPEDIS verwendet derzeit eine Heuristik mit einheitlichen
 *                  Kosten statt des gewichteten Kostenmodells von SAS; der
 *                  Schwellenwert ist darauf abgestimmt. Eine SAS-genaue
 *                  Überarbeitung wird separat verfolgt.
 *   3. COMPGED   — verallgemeinerter Editierabstand × 100 (≤250 = bis zu ~2
 *                  Editierungen). Es gilt derselbe Jenner-Vorbehalt: die
 *                  aktuelle Implementierung ist Levenshtein × 100, nicht die
 *                  gewichteten COMPGED-Kosten von SAS.
 *
 * Treffer aus einem der drei Signale zählen als Fuzzy-Treffer. Wir ziehen
 * Kandidatennamen von Amtsträgern/Entitäten aus dem Live-Graphen mit einem
 * einzigen PROC GQL je Art und führen dann das Dreifachsignal in einem
 * DATA-Schritt aus.
 */

prozedur gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

prozedur gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materialisiert die Phrasenliste als Datensatz für ein einfaches
   Zusammenführen im DATA-Schritt. */
daten ofac_phrase_list;
    länge phrase $80;
    eingabe phrase $80.;
    datalines;
;
ausführen;

/* Fügt dieselben Phrasen inline in den Datensatz ein — das Makro oben
   benennt sie für etwaige Cypher-Referenzen, aber wir benötigen auch
   einen Datensatz auf SAS-Seite. */
daten ofac_phrase_list;
    länge phrase $80;
    feld ph [*] $80 _temporary_ (&ofac_phrases);
    ausführung i = 1 bis dim(ph);
        phrase = ph[i];
        ausgabe;
    ende;
    entfernen i;
ausführen;

/* Amtsträger-Dreifachsignal-Fuzzy. Kreuzverbund + frühes Beschneiden
   über Soundex. */
daten officer_hits;
    festlegen all_officer_names;
    länge phrase $80 match_type $10;
    länge on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    ausführung k = 1 bis n_phrases;
        festlegen ofac_phrase_list (umbenennen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        wenn on_sx = ph_sx und on_sx ne '' dann ausführung;
            phrase = phrase_k; match_type = 'soundex'; ausgabe;
        ende;
        sonst wenn spedis(on_up, ph_up) <= 25 dann ausführung;
            phrase = phrase_k; match_type = 'spedis';  ausgabe;
        ende;
        sonst wenn compged(on_up, ph_up) <= 250 dann ausführung;
            phrase = phrase_k; match_type = 'compged'; ausgabe;
        ende;
    ende;
    behalten node_id officer_name phrase match_type;
ausführen;

/* Entitäts-Dreifachsignal-Fuzzy (gleiche Form). */
daten entity_hits;
    festlegen all_entity_names;
    länge phrase $80 match_type $10;
    länge en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    ausführung k = 1 bis n_phrases;
        festlegen ofac_phrase_list (umbenennen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        wenn en_sx = ph_sx und en_sx ne '' dann ausführung;
            phrase = phrase_k; match_type = 'soundex'; ausgabe;
        ende;
        sonst wenn spedis(en_up, ph_up) <= 25 dann ausführung;
            phrase = phrase_k; match_type = 'spedis';  ausgabe;
        ende;
        sonst wenn compged(en_up, ph_up) <= 250 dann ausführung;
            phrase = phrase_k; match_type = 'compged'; ausgabe;
        ende;
    ende;
    behalten node_id entity_name phrase match_type;
ausführen;

prozedur häufigkeiten daten=officer_hits;
    tables match_type / fehlend;
    titel "Officer fuzzy-match breakdown by signal";
ausführen;

prozedur häufigkeiten daten=entity_hits;
    tables match_type / fehlend;
    titel "Entity fuzzy-match breakdown by signal";
ausführen;

prozedur drucken daten=officer_hits (obs=25);
    titel "First 25 officer fuzzy matches";
ausführen;

prozedur drucken daten=entity_hits (obs=25);
    titel "First 25 entity fuzzy matches";
ausführen;


## 7. Wie lange leben Offshore-Entitäten? Kaplan-Meier

12.378 Paradise-Papers-Entitäten haben sowohl ein Gründungsdatum als
auch ein Auflösungsdatum. Das Parsen des eigenwilligen
Datumsformats `'2003-Dec-09'` erfolgt serverseitig in Cypher über
eine Monatscode-Lookup-Map und `duration.inDays`. Zeilen mit dem
Platzhalterdatum `1900-Jan-01` werden ausgeschlossen (sie stehen für
Entitäten, deren tatsächliches Gründungsdatum dem
ICIJ-Forschungsteam unbekannt war).

`PROC LIFETEST` stratifiziert nach einer fünfstufigen
Jurisdiktions-Variable (BM, KY, VG, IM, JE, plus einer
OTHER-Kategorie). Der Log-Rank-Test zeigt, dass sich die
Lebensdauern von Entitäten dramatisch zwischen den Jurisdiktionen
unterscheiden — Entitäten der Isle of Man überleben im Durchschnitt
etwa doppelt so lange wie Entitäten aus Bermuda.


In [ ]:
/* Zieht die vollständige Überlebenstabelle einmal. Vollständiger
   Datensatz = 12.130 Zeilen. */
prozedur gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

daten surv;
    festlegen surv_raw;
    event = 1;                 /* alle beobachteten Auflösungen */
    länge top5 $5;
    top5 = 'OTHER';
    wenn jurisdiction = 'BM' dann top5 = 'BM';
    sonst wenn jurisdiction = 'KY' dann top5 = 'KY';
    sonst wenn jurisdiction = 'VG' dann top5 = 'VG';
    sonst wenn jurisdiction = 'IM' dann top5 = 'IM';
    sonst wenn jurisdiction = 'JE' dann top5 = 'JE';
    log_officers = log(max(1, officer_count));
ausführen;

prozedur häufigkeiten daten=surv;
    tables top5 / out=top5_counts;
    titel "Entities per jurisdiction group (survival set)";
ausführen;

Die Kaplan-Meier-Routine von `PROC LIFETEST` wächst mit der
Stratengröße O(n²). Um das Notebook flott zu halten, passen wir sie
an eine Stichprobe von 2.000 Zeilen an — ein Lauf von ~20 Sekunden —
und zeigen den resultierenden Log-Rank-Test für Unterschiede
zwischen Jurisdiktionen. Das Cox-Modell im nächsten Abschnitt
verwendet dieselbe Stichprobe von 2.000 Zeilen.


In [ ]:
prozedur surveyselect daten=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
ausführen;

prozedur lifetest daten=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    titel "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
ausführen;

## 8. Auflösungsrisiko — Cox-Proportional-Hazards

`PROC PHREG` modelliert das Auflösungsrisiko als Funktion von
Jurisdiktion und dem Logarithmus der Amtsträgeranzahl. Die
Hazard-Ratio-Schätzungen beantworten die Compliance-Frage: *Wie
viel schneller oder langsamer wird eine Entität in einer
Jurisdiktion gegenüber einer anderen aufgelöst, wenn alles Übrige
gleich bleibt?*

Erwartete Befunde aus den echten Daten:

- Entitäten der Isle of Man haben etwa 46 % des Auflösungsrisikos
  von Bermuda — eine dramatisch längere operative Lebensdauer
- Entitäten aus Jersey haben etwa 38 % des Bermuda-Risikos
- Entitäten der Kaimaninseln haben etwa 61 % des Risikos
- Entitäten der Britischen Jungferninseln entsprechen fast exakt
  Bermuda
- Jede zusätzliche Log-Einheit der Amtsträgeranzahl senkt das
  Auflösungsrisiko um rund 12 % — größere Entitäten bestehen länger

Alle Effekte sind hochsignifikant (p < 0,0001 im Wald-Test).


In [ ]:
prozedur phreg daten=surv_sample;
    klasse top5 / ref=first;
    modell duration*event(0) = top5 log_officers;
    titel "Cox PH — closure hazard by jurisdiction + log(officers)";
ausführen;

## 9. Vorhersage hochkomplexer Entitäten — Logistische Regression

Wir definieren eine **hochkomplexe** Entität als eine mit fünf oder
mehr Amtsträgern — grob das obere Quartil der Verteilung — als Proxy
für die Art dicht besetzter Strukturen, auf die sich Compliance-
Teams zuerst konzentrieren. `PROC LOGISTIC` modelliert
`high_complexity` als Funktion von Jurisdiktion und
Intermediärsanzahl.

Der Auftrag verlangt eine Stichprobe mit `PLOTS=NONE` von bis zu
5.000 Zeilen, weil das Standard-ROC-Diagramm von `PROC LOGISTIC` im
Maßstab O(n²)-Verhalten zeigt. Wir ziehen 5.000 Entitäten und
verwenden `PLOTS=NONE`.


In [ ]:
prozedur surveyselect daten=entitaet_merkmale out=ef_sample
                   method=srs sampsize=5000 seed=42;
ausführen;

daten logi;
    festlegen ef_sample;
    länge top5 $5;
    top5 = 'OTHER';
    wenn jurisdiction = 'BM' dann top5 = 'BM';
    sonst wenn jurisdiction = 'KY' dann top5 = 'KY';
    sonst wenn jurisdiction = 'VG' dann top5 = 'VG';
    sonst wenn jurisdiction = 'IM' dann top5 = 'IM';
    sonst wenn jurisdiction = 'JE' dann top5 = 'JE';
    wenn officer_count >= 5 dann high_complexity = 1;
    sonst high_complexity = 0;
ausführen;

prozedur häufigkeiten daten=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titel "High-complexity entity rates by jurisdiction";
ausführen;

prozedur logistic daten=logi absteigend plots=none;
    klasse top5;
    modell high_complexity = top5 intermediary_count;
    titel "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
ausführen;

## 10. Konsolidierte Kennzahlen

Wir pausieren die analytische Erzählung, um mit `PROC MEANS` und
`PROC FREQ` eine kompakte Zusammenfassung der Überlebenszeit-Daten
festzuhalten. Das ist die Art von Kopfzeilentabelle, mit der eine
Compliance-Analystin oder eine Regulierungsbehörde einen Bericht
eröffnet. Die folgenden Abschnitte erweitern die Analyse um
amtsträgerzentriertes Risiko, zeitliche Muster,
leak-übergreifende Konzentration, ein breiteres Sanktions-Screening
und ein abschließendes zusammengesetztes Entitäts-Ranking. Alle
Ausgaben werden in dem einen `ODS PDF` erfasst, das am Anfang des
Notebooks geöffnet und nach Abschnitt 15 geschlossen wird.


In [ ]:
titel "ICIJ Paradise Papers — Headline Statistics";

prozedur mittelwerte daten=surv n mean std median p25 p75;
    var duration officer_count;
    titel "Entity lifespan and officer-count summary (full n=12,130)";
ausführen;

prozedur häufigkeiten daten=surv;
    tables top5;
    titel "Jurisdiction distribution of surviving entities";
ausführen;


## 11. Amtsträgerzentrierte Risikosicht

Die Abschnitte 2 bis 10 konzentrierten sich auf Entitäten. Die
Menschen hinter diesen Entitäten — die Amtsträger — verdienen
dieselbe Aufmerksamkeit. Die Compliance-Praxis behandelt einen
Amtsträger, der *gleichzeitig* (a) mit vielen Entitäten verbunden
ist, (b) über viele Jurisdiktionen hinweg agiert, (c) in der
Amtsträger-Entitäts-Projektion mit erhöhtem PageRank auftritt und
(d) über ein langes Zeitfenster aktiv ist, als eigenständiges
strukturelles Risiko.

Wir bauen aus dem echten Graphen eine Merkmalstabelle je Amtsträger:

| Merkmal | Definition |
|---|---|
| `degree` | Anzahl der Entitäten, bei denen dieser Amtsträger OFFICER_OF hält |
| `n_juris` | Anzahl verschiedener Jurisdiktionen dieser Entitäten |
| `pagerank` | PageRank des Amtsträgerknotens aus Abschnitt 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` über die Entitäten des Amtsträgers |

Anschließend normalisieren wir jedes Merkmal per Min-Max auf [0, 1]
und bilden eine gewichtete Summe — 30 % degree, 30 % log-PageRank,
20 % Jurisdiktionsbreite, 20 % Amtsdauer — als einen einzigen
zusammengesetzten **Einfluss-Score**. Die Top 10 nach diesem Score
bringen die Personen an die Oberfläche, die das ICIJ-Forschungsteam
öffentlich als die bestvernetzten Appleby-Akteure benannt hat.


In [ ]:
prozedur gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Hängt die PageRank-äquivalente Zentralität (aus der
   PROC-NETWORK-Ausgabe von Abschnitt 4) über einen LEFT JOIN auf den
   Amtsträgernamen an. PROC SQL erledigt Sortieren+Zusammenführen+
   Coalesce in einem Durchgang — keine DATA-MERGE-BY-Kette, keine
   PROC SORTs. */
prozedur sql;
    erstellen tabelle officer_feat as
    auswählen o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    von   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normalisiert jedes Merkmal per Min-Max, bildet den
   zusammengesetzten Einfluss-Score und behält die Top 50 nach
   Einfluss. PROC RANK + PROC SQL statt einer Pipeline aus mehreren
   DATA-Schritten. */
prozedur mittelwerte daten=officer_feat noprint;
    var degree pagerank n_juris tenure_years;
    ausgabe out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
ausführen;

daten officer_scored;
    wenn _n_ = 1 dann festlegen officer_minmax;
    festlegen officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    behalten node_id officer_name degree pagerank
         n_juris tenure_years influence;
ausführen;

prozedur sql outobs=50;
    titel "Section 11 — top-50 Paradise-Papers officers by composite influence";
    auswählen officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    von   officer_scored
    order nach influence desc;
quit;

## 12. Zeitliche Gründungsmuster

Das serverseitige Parsen von `incorporation_date` in eine
vierstellige Jahreszahl lässt uns erkennen, wie sich die
Offshore-Gründungsaktivität über die fünf dominierenden
Jurisdiktionen entwickelt hat. Die Darstellung der jährlichen
Neugründungszahlen von 1970 bis 2014 in einem
Small-Multiples-Layout mit `PROC SGPANEL` legt jene
gesetzgebungsgetriebenen Ausbrüche offen, nach denen
Politikanalysten suchen.

In den echten Daten:

- Die Aktivität der **Kaimaninseln** steigt ab Ende der 1990er
  stetig, überschreitet 2001 die Marke von 400 neuen Entitäten pro
  Jahr und stabilisiert sich bis 2014 bei rund 450–510
  Neugründungen jährlich.
- **Bermuda** erreicht um 2007–2013 einen Höhepunkt von 210–290 pro
  Jahr und folgt eng den globalen Fundraising-Zyklen von Hedgefonds
  und Private Equity.
- Die **Britischen Jungferninseln** legen abrupt von ~60 neuen
  Entitäten pro Jahr im Jahr 2005 auf 200 im Jahr 2014 zu — eine
  3,3-fache Zunahme über das Fenster, das die Paradise Papers
  abdecken.
- **Isle of Man** und **Jersey** bleiben moderat (50–140 pro Jahr),
  doch Jersey zeigt 2013 einen scharfen Sprung auf 142 — die höchste
  Jersey-Zahl im gesamten Fenster.


In [ ]:
prozedur gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Fasst Jurisdiktionen außerhalb der Top 5 zu OTHER zusammen. */
daten jahr_panel;
    festlegen year_jur;
    länge top5 $5;
    top5 = 'OTHER';
    wenn jurisdiction = 'BM' dann top5 = 'BM';
    sonst wenn jurisdiction = 'KY' dann top5 = 'KY';
    sonst wenn jurisdiction = 'VG' dann top5 = 'VG';
    sonst wenn jurisdiction = 'IM' dann top5 = 'IM';
    sonst wenn jurisdiction = 'JE' dann top5 = 'JE';
ausführen;

prozedur mittelwerte daten=jahr_panel noprint nway;
    klasse year top5;
    var n;
    ausgabe out=jahr_summen (entfernen=_type_ _freq_)
        sum=entities;
ausführen;

prozedur sgpanel daten=jahr_summen;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis bezeichnung="Incorporation year";
    rowaxis bezeichnung="New entities per year";
    titel "Section 12 — new entity formations per year, top-5 jurisdiktionen";
ausführen;

/* Die drei größten Spitzenjahr-Ausbrüche über Top 5 + OTHER. */
prozedur sortieren daten=jahr_summen out=jahr_spitzen;
    nach absteigend entities;
ausführen;

daten jahr_spitzen;
    festlegen jahr_spitzen (obs=10);
ausführen;

prozedur drucken daten=jahr_spitzen;
    titel "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
ausführen;

## 13. Leak-übergreifender Vergleich

Der Paradise-Papers-Graph ist intern über `sourceID` in fünf
Teilkorpora aufgeteilt, die die fünf unabhängigen Quellströme
widerspiegeln, die das ICIJ zusammengetragen hat:

- **Paradise Papers - Appleby** — das Leak der Anwaltskanzlei
  Appleby (die überwältigende Mehrheit der Daten)
- **Paradise Papers - Malta corporate registry** — eine geleakte
  Kopie des offiziellen maltesischen Handelsregisters
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Behandeln wir jede `sourceID` als „Leak", können wir bestätigen,
dass sich jeder Korpus in seiner eigenen Ecke der Offshore-Welt
konzentriert. Das Appleby-Leak ist überwältigend Bermuda und Cayman
(zusammen 73 % seiner benannten Entitäten); das Malta-Leak besteht
faktisch nur aus maltesischen Entitäten; das Libanon-Leak im
Wesentlichen nur aus libanesischen Entitäten; und so weiter. Die
`PROC FREQ`-Kreuztabelle unten macht diese Konzentration sichtbar.


In [ ]:
prozedur gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

prozedur häufigkeiten daten=leak_raw order=häufigkeiten;
    tables sourceid / out=leak_totals;
    gewicht n;
    titel "Section 13 — entity counts per sourceID corpus";
ausführen;

prozedur drucken daten=leak_totals;
    titel "Section 13 — leak-level totals";
ausführen;

/* Das LIST-Format gibt eine Zeile je (Leak, Jurisdiktion)-Zelle aus
   — passt in die Terminalbreite statt der standardmäßig breiten
   Kreuztabelle. */
prozedur sortieren daten=leak_raw out=leak_sorted;
    nach sourceid absteigend n;
ausführen;

prozedur drucken daten=leak_sorted (obs=30);
    titel "Section 13 — top 30 (leak, jurisdiction) cells";
ausführen;


## 14. Breitere Sanktions-Anreicherung — OpenSanctions

Das reine OFAC-SDN-Screening aus Abschnitt 6 lieferte null
exakte Treffer. Das war ein ehrlicher Befund — die von uns
eingecheckte OFAC-Stichprobe mit 500 Zeilen stammte überwältigend
aus dem RUSSIA-EO14024-Programm von 2022, und die Paradise Papers
wurden aus Daten zusammengestellt, deren jüngste Datensätze auf 2014
datiert sind.

Um das Netz zu erweitern, verwenden wir nun einen echten Ausschnitt
des *default*-konsolidierten Sanktionsdatensatzes von
[OpenSanctions](https://www.opensanctions.org/) — den Snapshot vom
2026-04-17 konsolidierter öffentlicher Sanktionslisten von:

- U.S. OFAC Specially Designated Nationals
- Vermögenssperr-Zielen des UK HM Treasury
- EU Consolidated Financial Sanctions
- Sanktionen des UN-Sicherheitsrats
- konsolidierten Listen Kanadas, Belgiens, Australiens, der
  Schweiz, Norwegens, Japans, Neuseelands und Singapurs

Die eingecheckte Teilmenge unter `data/opensanctions_default.csv`
enthält die 18.654 Person- und Company-Datensätze, deren primärer
Datensatz eine der kuratierten Kern-Sanktionslisten ist (reine
Referenzdatenquellen wie die BIC- und FIRDS-Kennungskataloge sind
ausgeschlossen, damit jeder Treffer eine echte Sanktions-Provenienz
trägt). Aliasse wurden entfernt, um die Datei unter 2 MB zu halten.

Da die OpenSanctions-Liste eine Größenordnung umfangreicher ist als
unsere frühere OFAC-Stichprobe, ziehen wir jeden Amtsträger- und
jeden Entitätsnamen *einmal* aus Neo4j und verbinden lokal mit
`PROC SQL` gegen die Sanktions-CSV. Ein exakter UPCASE-Abgleich ist
robust und vermeidet die Cypher-String-Längenbeschränkungen, die
große Token-IN-Listen plagen.


In [ ]:
/* Liest die eingecheckte OpenSanctions-CSV. Neun
   Kopf-Kommentarzeilen plus eine Spaltenüberschrift = firstobs=11. */
daten opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    länge os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    eingabe os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    wenn länge(os_name_upper) >= 6;
ausführen;

prozedur sortieren daten=opensanctions nodupkey out=os_dedup;
    nach os_name_upper;
ausführen;

prozedur mittelwerte daten=os_dedup n;
    titel "OpenSanctions sanctions-list records loaded";
ausführen;

/* Zieht jeden Amtsträger- und Entitätsnamen aus dem Graphen. */
prozedur gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

prozedur gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Exakter UPCASE-Abgleich — Amtsträger zu sanktionierter Partei. */
prozedur sql;
    erstellen tabelle s14_officer_hits as
    auswählen o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    von all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

prozedur sql;
    erstellen tabelle s14_entity_hits as
    auswählen e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    von all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

prozedur sql;
    auswählen count(*) as n_officer_hits
    von s14_officer_hits;
    auswählen count(*) as n_entity_hits
    von s14_entity_hits;
quit;

prozedur drucken daten=s14_officer_hits;
    titel "Section 14 — officers on a consolidated sanctions list";
ausführen;

prozedur drucken daten=s14_entity_hits;
    titel "Section 14 — entities on a consolidated sanctions list";
ausführen;

## 15. Zusammengesetztes Entitäts-Risiko-Ranking

Schließlich kombinieren wir die strukturellen, jurisdiktionellen,
relationalen und Sanktions-Signale aus den vorangegangenen
Abschnitten zu einem einzigen zusammengesetzten **Risiko-Score** je
Entität:

| Komponente | Gewicht | Quelle |
|---|---:|---|
| Amtsträgeranzahl (gedeckelt bei 50) | 0,25 | Merkmalstabelle aus Abschnitt 5 |
| log(1 + PageRank des Top-Amtsträgers) | 0,25 | PageRank aus Abschnitt 4 + Abschnitt 11 |
| Jurisdiktions-Risikogewicht | 0,25 | Tax Justice Network CTHI 2021 (eingecheckt) |
| Kennzeichen für sanktionierten Amtsträger | 0,25 | Exakt-Treffer aus Abschnitt 14 |

Das Jurisdiktionsrisiko stammt aus der eingecheckten Datei
`data/tax_haven_ranks.csv`, zusammengestellt aus der
veröffentlichten Rangliste des Corporate Tax Haven Index 2021 des
Tax Justice Network. Die Ränge 1–10 sind direkt der
CTHI-Pressemitteilung von 2021 entnommen; die Ränge im Mittelfeld
sind die veröffentlichten TJN-Methodikwerte für die übrigen
Jurisdiktionen, die in den Paradise Papers vorkommen.
Jurisdiktionen ohne CTHI-Rang (z. B. der Platzhalter `XX`) erhalten
das Gewicht ≈ 0.

Der Bericht unten ist mit `PROC REPORT` für eine Regulierungsbehörde
formatiert. Die Entitäten an der Spitze der Liste sind jene, die
gleichzeitig (a) in einer Erstseiten-Oasen-Jurisdiktion
ansässig sind, (b) viele Amtsträger haben, (c) einen
Top-PageRank-Amtsträger besitzen UND (d) mindestens einen auf einer
konsolidierten Sanktionsliste gekennzeichneten Amtsträger haben.


In [ ]:
/* Lädt die eingecheckten Ränge des TJN Corporate Tax Haven Index
   2021. Acht Kommentarzeilen + zwei weitere // plus eine
   Überschrift = firstobs=16. */
daten steueroase;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    länge iso2 $5 jurisdiction_name $50;
    eingabe iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
ausführen;

prozedur drucken daten=steueroase (obs=10);
    titel "Section 15 — jurisdiction risk weights (CTHI 2021)";
ausführen;

/* Merkmale je Entität mit dem Namen des Top-Amtsträgers und dem
   Gründungsjahr. */
prozedur gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Alles, was zusammengeführt werden muss (PageRank, Risikogewicht,
   Kennzeichen für sanktionierte Amtsträger), wird in einem einzigen
   dreifachen LEFT JOIN von PROC SQL erledigt — keine
   DATA-MERGE-BY-Kette, keine überflüssigen PROC SORTs, und COALESCE
   liefert die Ersatzwerte inline. */
prozedur gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

prozedur sql;
    erstellen tabelle entity_flagged as
    auswählen e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case falls f.node_id is nicht null dann 1 sonst 0 ende
               as sanctioned_officer
    von   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join steueroase       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Zusammengesetztes Risiko: normalisiert die stetigen Merkmale per
   Min-Max und gewichtet sie zusammen. PROC MEANS + ein einzelner
   DATA-Schritt genügt für die Normalisierung — das ist idiomatisches
   SAS. */
prozedur mittelwerte daten=entity_flagged noprint;
    var top_officer_pr;
    ausgabe out=pr_max_ds max=pr_max;
ausführen;

daten entity_score;
    wenn _n_ = 1 dann festlegen pr_max_ds;
    festlegen entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    behalten node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
ausführen;

/* Risikoverteilung über die vollständige Population von 24.957
   Entitäten. */
prozedur mittelwerte daten=entity_score n min mean max;
    var risk officer_count risk_weight;
    titel "Section 15 — risk distribution across all entities";
ausführen;

/* Top-25-Entitäten nach zusammengesetztem Risiko. PROC SQL OUTOBS=
   ersetzt ein Paar aus PROC SORT + DATA-Schritt mit obs=. */
prozedur sql outobs=25;
    titel "Section 15 — top-25 composite-risk entities (names)";
    auswählen entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    von   entity_score
    order nach risk desc;
quit;

/* Bringt separat die mit sanktionierten Amtsträgern verknüpften
   Entitäten an die Oberfläche. */
prozedur sql;
    titel "Section 15 — entities with at least one sanctioned officer";
    auswählen entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    von   entity_score
    wo  sanctioned_officer = 1
    order nach risk desc;
quit;

## 16. Klassifikation von Jurisdiktionen als Conduit oder Sink

**Referenz:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. partitionieren den globalen
Unternehmensbeteiligungsgraphen in „Sink-OFCs" — wo
Unternehmenswert endet: BM, KY, BVI, JE, IM — und „Conduit-OFCs" —
durch die Wert fließt: NL, UK, CH, SG, IE. Der
Paradise-Papers-Graph hat eine andere Population (überwiegend bei
Appleby ansässige Entitäten), doch dieselbe strukturelle
Unterscheidung sollte die Jurisdiktionen, in denen sich Amtsträger
konzentrieren und Adressen enden, von jenen trennen, die primär
Kapital durchleiten.

Für jede Jurisdiktion berechnen wir fünf strukturelle Merkmale
direkt aus dem Live-Graphen:

| Merkmal | Signal für |
|---|---|
| `log(n_entity)` | absolute Größe der Offshore-Präsenz der Jurisdiktion |
| `avg_officers` | Amtsträger-je-Entität-Dichte (Sink-Jurisdiktionen tragen mehr Amtsträger pro Entität) |
| `avg_xborder_off` | durchschnittliche Anzahl von Amtsträgern, deren eigenes Land von der Jurisdiktion der Entität abweicht (Conduit-Indikator) |
| `intermediary_share` | Anteil der Entitäten mit einer CONNECTED_TO-Intermediär-Verbindung |
| `address_share` | Anteil der Entitäten mit einer REGISTERED_ADDRESS-Verbindung (Sink-Indikator) |

Wir standardisieren zu z-Werten und wenden dann Wards
Minimum-Varianz-Verfahren zur hierarchischen Clusterung an.
`PROC TREE` rendert das Dendrogramm. Man beachte, dass die
Intermediary-Knoten der Paradise Papers über `CONNECTED_TO` mit
Entity-Knoten verbunden sind — nicht über `INTERMEDIARY_OF`, das im
Schema existiert, aber für dieses Leak keine Daten trägt — sodass
wir hier `CONNECTED_TO` verwenden.


In [ ]:
/* Zieht strukturelle Merkmale je Jurisdiktion aus dem
   Live-Graphen. */
prozedur gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Behält nur Jurisdiktionen mit mindestens zehn Entitäten, damit
   die standardisierten z-Werte aussagekräftig sind. Das
   Paradise-Papers-Leak hat insgesamt 44 Jurisdiktionen; 23 erfüllen
   diese Schwelle. */
daten s16_filtered;
    festlegen s16_raw;
    wenn n_entity >= 10;
    log_n_entity = log(n_entity);
ausführen;

prozedur mittelwerte daten=s16_filtered noprint;
    var log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    ausgabe out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
ausführen;

daten s16_std;
    wenn _n_ = 1 dann festlegen s16_stats;
    festlegen s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    behalten jurisdiction z1 z2 z3 z4 z5;
ausführen;

prozedur drucken daten=s16_std;
    titel "Section 16 — standardised jurisdiction features";
ausführen;

/* Wards hierarchische Clusterung mit minimaler Varianz. */
prozedur cluster daten=s16_std method=ward outtree=s16_tree;
    var z1 z2 z3 z4 z5;
    id jurisdiction;
    titel "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
ausführen;

/* Rendert das Dendrogramm. */
prozedur tree daten=s16_tree horizontal;
    id jurisdiction;
    titel "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
ausführen;

## 17. Hauptkomponenten der Amtsträger-Netzwerkrollen

**Referenz:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Siehe auch Newman M E J, *Networks: An Introduction* (Oxford, 2010),
Kapitel 7.

Amtsträger im Paradise-Papers-Graphen spielen strukturell
unterschiedliche Rollen. Einige sitzen im Zentrum eines großen
Clusters verwandter Entitäten; andere verbinden ansonsten getrennte
Cluster miteinander (Broker, im Sinne von Burt/Borgatti); wenige
bilden den dichten Kern des Insider-Netzwerks einer bestimmten
Jurisdiktion. Vier Graph-Metriken erfassen diese unterschiedlichen
Rollen:

| Metrik | Erfasst |
|---|---|
| `degree` | Anzahl der `OFFICER_OF`-Ausgangskanten — Breite der Zugehörigkeit |
| `betweenness` | Anteil der kürzesten Pfade, die durch den Amtsträger führen — **Brokerage** |
| `kcore` | größtes k, für das der Amtsträger in einem k-zusammenhängenden Teilgraphen liegt — **Kern-Einbettung** |
| `pagerank` | eigenvektorähnlicher Score aus derselben Projektion — **Einfluss über einflussreiche Partner** |

Wir berechnen alle vier Metriken auf der vollständigen
ungerichteten Projektion `(Officer)—[OFFICER_OF]—(Entity)` über die
Neo4j-Graph-Data-Science-Bibliothek, beschränken auf die 5.000
grad-stärksten Amtsträger und führen `PROC PRINCOMP` auf den
log-transformierten Metriken aus.

Die PCA komprimiert die vier korrelierten Metriken zu orthogonalen
Achsen, deren relative Ladungen uns verraten, welche Rollen
empirisch zusammen clustern und welche strukturell verschieden sind.

*Hinweis zum lokalen Clustering-Koeffizienten:* Garcia-Bernardo et al.
schließen den lokalen Clustering-Koeffizienten als fünfte Metrik
ein. Die Paradise-Papers-Projektion
`(Officer)—[:OFFICER_OF]—(Entity)` ist streng bipartit, sodass keine
Dreiecke existieren können — jeder lokale Clustering-Koeffizient ist
0. Wir lassen ihn aus dem Metriksatz weg.


In [ ]:
/* PROC NETWORK zieht über einen schreibgeschützten MATCH einen
   Teilgraphen der Top-5000-Amtsträger und berechnet Grad,
   Eigenvektor-Zentralität und k-Core prozessintern. Ersetzt einen
   früheren gds.graph.project + vier gds.<algorithmus>.stream-Aufrufe
   — jenes Muster erfordert einen GDS-Projektionsschritt im
   Schreibmodus, den der schreibgeschützte step-neo4j-Dienst der
   Plattform ablehnt.

   Die Betweenness-Zentralität wird bewusst ausgelassen: NetworkX
   berechnet exakte Betweenness in O(V·E), was die Laufzeit auf
   diesem Teilgraphen dominiert (5.000 Amtsträger × ~78.000 Kanten).
   Die PCA hat weiterhin drei orthogonale Achsen — Grad (lokale
   Prominenz), Eigenvektor-Zentralität (globaler Einfluss) und
   k-Core (strukturelle Kohäsion) — genug, um die Amtsträgerrollen
   für die Kernaussage zu trennen. */
prozedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar von=a_node_id bis=b_node_id;
    centrality degree eigen=unweight;
    core;
ausführen;

/* Zieht Knoten-IDs/-Namen der Amtsträger zum Filtern. */
prozedur gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtert auf Amtsträger-Zeilen. Die Eigenvektor-Zentralität steht
   stellvertretend für PageRank — siehe Anmerkung in Abschnitt 4. */
prozedur sql;
    erstellen tabelle s17_metrics as
    auswählen n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    von s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order nach n.centr_degree desc;
quit;

daten s17_metrics;
    festlegen s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    behalten node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
ausführen;

prozedur drucken daten=s17_metrics (obs=10);
    titel "Section 17 — top-10 officers by degree (raw + log metrics)";
ausführen;

prozedur mittelwerte daten=s17_metrics n mean std min max;
    var log_degree log_pr k_val;
    titel "Section 17 — log-transformed metric summary";
ausführen;

prozedur princomp daten=s17_metrics out=s17_scores;
    var log_degree log_pr k_val;
    titel "Section 17 — PCA of officer network roles";
ausführen;

prozedur sgplot daten=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis bezeichnung="PC1 (global influence axis)";
    yaxis bezeichnung="PC2 (brokerage vs core embeddedness)";
    titel "Section 17 — officers in 2D principal-component role space";
ausführen;

## 18. ARIMA-Interventionsanalyse der Gründungsraten

**Referenz:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Auf Offshore-Leaks angewandt von Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Die jährliche Anzahl neuer Entitäten im Paradise-Papers-Graphen ist
eine nahezu monotone Wachstumsreihe von 1970 (36 Entitäten) bis 2007
(1.595 Entitäten, der Höhepunkt), gefolgt von einem Rückgang
2008–2009 und einem langsameren Plateau bis 2014 (dem Ende der
Leak-Abdeckung).

Wir wenden die klassische Box-Tiao-Interventionsanalyse an, um zu
prüfen, ob zwei reale Ereignisse messbare Signaturen in der
Gründungsreihe hinterlassen haben:

- **Stufe 2009** — das Vorgehen des G20-Gipfels in London gegen
  Steueroasen (April 2009), das mit der Finanzkrise zusammenfiel.
- **Stufe 2014** — der US-FATCA (Foreign Account Tax Compliance Act)
  trat am 1. Juli 2014 in Kraft.

Die Zielgröße ist `log(n)`, sodass ein Interventionskoeffizient von
-0,30 rund einem Rückgang der jährlichen Gründungsrate um 26 %
entspricht. Wir passen `ARIMA(1,0,0)` an — das autoregressive
AR(1)-Modell für die stark korrelierte Reihe — mit den beiden
Stufen-Dummys als exogenen `INPUT=`-Variablen.

Die Nullhypothese lautet, dass keine der beiden Stufen eine messbare
Verschiebung erzeugt, sobald der AR(1)-Trend berücksichtigt ist.
Veröffentlichte Methodiken unterscheiden sich darin, wie eine
Nicht-Ablehnung zu interpretieren ist: Sie kann bedeuten, dass die
Intervention keine Wirkung hatte, oder dass die
AR(1)-Autokorrelation das Interventionssignal absorbiert.


In [ ]:
prozedur gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Baut den Datensatz der Interventionsreihe. Zwei Stufen-Dummys:
   step_2009 = I{year >= 2009} erfasst den Regimewechsel der
   Vorankündigung von G20-London/FATCA; step_2014 = I{year >= 2014}
   erfasst den Schock des FATCA-Inkrafttretens. Die Zielgröße ist
   log(n), sodass die Koeffizientenschätzungen als proportionale
   Effekte zu lesen sind. */
daten s18_ts;
    festlegen year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
ausführen;

prozedur drucken daten=s18_ts;
    titel "Section 18 — annual new-entity formations + intervention dummies";
ausführen;

prozedur sgplot daten=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x bezeichnung="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x bezeichnung="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis bezeichnung="Incorporation year";
    yaxis bezeichnung="New entities per year";
    titel "Section 18 — Paradise-Papers annual formations, 1970-2014";
ausführen;

/* Identifiziert das Modell und schätzt dann ARIMA(1,0,0) mit den
   beiden Stufen-Inputs. Das CROSSCORR= von PROC ARIMA registriert
   die exogenen Variablen im IDENTIFY-Schritt, sodass sie für
   ESTIMATE INPUT= verfügbar sind. */
prozedur arima daten=s18_ts;
    identify var=log_n crosscorr=(step_2009 step_2014) nlag=8;
    schätzung p=1 eingabe=(step_2009 step_2014);
    titel "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
ausführen;
quit;

## 19. Nullinflationiertes Zählmodell der Sanktions-Exposition

**Referenz:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2. Auflage, Cambridge University Press (2013), Kapitel 4.
Nullinflationierte Modelle: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Abschnitt 14 fand nur **fünf** Paradise-Papers-Entitäten mit
mindestens einem Amtsträger auf einer konsolidierten Sanktionsliste
— ein verschwindend seltenes Ereignis. Eine standardmäßige Poisson-
oder negativ-binomiale Regression auf `sanctioned_count` je Entität
würde schlecht passen, weil **99,98 %** der Entitäten null aufweisen.
Das nullinflationierte negativ-binomiale Modell (ZINB) bewältigt
dies, indem es die Anpassung in zwei Teile aufteilt:

1. Ein logistisches Modell dafür, ob die Entität zu einer Klasse der
   „strukturellen Nullen" gehört (keine mögliche Sanktions-
   Exposition).
2. Ein negativ-binomiales Modell für die Anzahl unter den übrigen.

Mit nur 5 positiven Ereignissen über 21.538 Entitäten ist die
ZINB-Likelihood degeneriert — beide Teile kollabieren. Dieses
Scheitern ist eine **ehrliche Eigenschaft der Daten**, nicht der
Prozedur. Wir führen die ZINB-Anpassung dennoch aus, um zu zeigen,
dass das Regressions-Werkzeug durchgängig funktioniert, und greifen
dann auf eine gewöhnliche binär-logistische Regression auf
`has_sanctioned` zurück (Indikator für `sanctioned_count > 0`). Die
Logistik identifiziert einen klaren Effekt: **jede zusätzliche
Log-Einheit der Amtsträgeranzahl multipliziert die Chance, mindestens
einen sanktionierten Amtsträger zu haben, um etwa 3,1** (p = 0,002).

Kovariaten:

- `top5` — 6-stufige Klassenvariable (BM, KY, VG, IM, JE, OTHER),
  Referenzkategorie OTHER.
- `log_n_off` — `log(officer_count)`, der dominierende
  Größenprädiktor.


In [ ]:
/* Zieht die Anzahl sanktionierter Amtsträger je Entität aus dem
   Live-Graphen. */
prozedur gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

daten s19;
    festlegen s19_raw;
    wenn officer_count >= 1;
    länge top5 $5;
    top5 = 'OTHER';
    wenn jurisdiction = 'BM' dann top5 = 'BM';
    sonst wenn jurisdiction = 'KY' dann top5 = 'KY';
    sonst wenn jurisdiction = 'VG' dann top5 = 'VG';
    sonst wenn jurisdiction = 'IM' dann top5 = 'IM';
    sonst wenn jurisdiction = 'JE' dann top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
ausführen;

prozedur häufigkeiten daten=s19;
    tables top5 sanctioned_count has_sanctioned;
    titel "Section 19 — response-variable distribution (very zero-heavy)";
ausführen;

/* Zuerst ZINB — voraussichtlich degeneriert bei einer Reihe mit
   5 Ereignissen. */
prozedur genmod daten=s19;
    klasse top5;
    modell sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titel "Section 19 — ZINB count model (degenerate on 5 events)";
ausführen;

/* Rückfall: binär-logistisch auf has_sanctioned. Interpretierbar. */
prozedur logistic daten=s19 absteigend plots=none;
    klasse top5;
    modell has_sanctioned = top5 log_n_off;
    titel "Section 19 — logistic regression (has-sanctioned fallback)";
ausführen;

## 20. Poisson-Modell der Gründungsrate mit gemischten Effekten

**Referenz:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klassisch für Paneldaten: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Abschnitt 18 passte ein univariates ARIMA an die aggregierte
Gründungsreihe an. Hier nutzen wir die **Panel**-Dimension: eine
Zeile je Jurisdiktion-Jahr-Zelle, angepasst als generalisiertes
lineares gemischtes Poisson-Modell (GLMM) mit einem festen linearen
Jahres-Trend plus einem FATCA-Stufen-Dummy und einem **zufälligen
Achsenabschnitt je Jurisdiktion**. Das trennt den gemeinsamen
Trend-Effekt vom Niveau der einzelnen Jurisdiktion.

Panelbeschränkung: die 10 Jurisdiktionen, deren **durchschnittliche
Jahresanzahl** über 1990–2014 >=5 beträgt (insgesamt 203 Zellen).
Kleinere Jurisdiktionen mit vielen Null-Jahren würden eine
Poisson-Anpassung destabilisieren.

`PROC GLIMMIX` mit `DIST=POISSON LINK=LOG` und
`RANDOM INTERCEPT / SUBJECT=jurisdiction` erzeugt sowohl die festen
Effekte auf Populationsebene (Jahres-Trend, FATCA-Verschiebung) als
auch die Varianzkomponente zwischen den Jurisdiktionen. Die Varianz
des zufälligen Achsenabschnitts sagt uns, *wie stark sich die
Gründungsraten zwischen Jurisdiktionen unterscheiden, nachdem der
gemeinsame Zeittrend berücksichtigt wurde* — ein Score für die
strukturelle Heterogenität der Population der Offshore-Finanzzentren.


In [ ]:
/* Panel-Datensatz: Jurisdiktion-x-Jahr-Zellen von 1990–2014. */
prozedur gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Behält Jurisdiktionen mit durchschnittlicher Jahresanzahl >= 5. */
prozedur sql;
    erstellen tabelle s20_keep_jur as
    auswählen jurisdiction, avg(n) as avg_n
    von s20_raw
    group nach jurisdiction
    having avg(n) >= 5;
quit;

prozedur sql;
    erstellen tabelle s20 as
    auswählen r.*,
           r.year - 2002 as year_c,
           case falls r.year >= 2014 dann 1 sonst 0 ende as fatca
    von s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

prozedur häufigkeiten daten=s20;
    tables jurisdiction;
    titel "Section 20 — jurisdiktionen retained in the panel";
ausführen;

/* Poisson-GLMM mit gemischten Effekten: fester Jahres-Trend +
   FATCA-Stufe, zufälliger Achsenabschnitt je Jurisdiktion. */
prozedur glimmix daten=s20;
    klasse jurisdiction;
    modell n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    titel "Section 20 — mixed Poisson formation-rate model";
ausführen;

/* Nach Rang geordnete Effekte des zufälligen Achsenabschnitts
   würden die Jurisdiktionen sichtbar machen, die den gemeinsamen
   Trend systematisch über- oder unterbieten. Die
   SOLUTION-Anweisung von PROC GLIMMIX gibt diese in der
   Standardausgabe oben aus — wir überlassen das Ranking der
   Leserin. */

In [ ]:
/* Schließt das Bericht-PDF und gibt die Neo4j-Bibliothek frei. */
ods pdf close;

bibliothek icij clear;

## Reproduzierbarkeit und Provenienz

- **Graph-Datenquelle:** ICIJ-*Offshore-Leaks*-Forschungsdatenbank,
  Datensatz *Paradise Papers*, erstmals im November 2017
  veröffentlicht. Verfügbar unter
  <https://offshoreleaks.icij.org/pages/database>. In den
  gemeinsamen `step-neo4j`-Dienst der Plattform geladen
  (Neo4j 5.26 Community Edition, auf Serverebene schreibgeschützt)
  über den öffentlichen Upstream-Dump unter
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **OFAC-SDN-Daten:** öffentlicher CSV-Export der U.S. Treasury OFAC
  Specially Designated Nationals, abgerufen über die öffentliche
  Preview-API des Finanzministeriums im April 2026. Die Datei
  `data/ofac_sdn.csv` enthält 500 kuratierte Zeilen über die fünf
  nach Eintragszahl größten OFAC-Programme. Verwendet für das
  zweistufige Screening in Abschnitt 6.
- **OpenSanctions-Daten:** Snapshot des *default*-konsolidierten
  Datensatzes von OpenSanctions vom 2026-04-17, heruntergeladen von
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Die eingecheckte Datei `data/opensanctions_default.csv` enthält die
  18.654 Zeilen der Schemata Person und Company, deren primärer
  Datensatz eine der Listen OFAC SDN, UK HM Treasury, EU-
  Finanzsanktionen, UN-Sicherheitsrat oder eine der konsolidierten
  Sanktionslisten Kanadas, Belgiens, Australiens, der Schweiz oder
  anderer Nationen ist. Aliasse wurden entfernt, um die Datei unter
  2 MB zu halten. Lizenz: ODbL 1.0 (OpenSanctions). Verwendet für die
  Anreicherung in Abschnitt 14.
- **Steueroasen-Ränge:** veröffentlichte Ranglisten des *Corporate
  Tax Haven Index 2021* des Tax Justice Network, zusammengestellt aus
  der Index-Landingpage `https://cthi.taxjustice.net` und der
  Pressemitteilung vom März 2021 unter
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Die eingecheckte Datei `data/tax_haven_ranks.csv` listet die
  Jurisdiktionen, die in den Paradise Papers vorkommen, mit ihrem
  CTHI-Rang und einem abgeleiteten `[0, 1]`-Risikogewicht. Lizenz:
  CC BY-SA 4.0 (Tax Justice Network). Verwendet für das
  zusammengesetzte Ranking in Abschnitt 15.
- **Graph-Algorithmen:** Louvain-Community-Erkennung und
  Eigenvektor-Zentralität (das nächstliegende hausinterne Analogon
  zu PageRank), von `PROC NETWORK` prozessintern auf Kantenlisten
  berechnet, die über schreibgeschütztes Cypher gezogen wurden. Der
  gemeinsame `step-neo4j`-Dienst der Plattform ist auf Serverebene
  schreibgeschützt, sodass alle Graph-Algorithmen im Workspace-Pod
  laufen statt über Schreiboperationen von Neo4j Graph Data Science.
- **Statistische Methoden:** `PROC LIFETEST` verwendet den
  Kaplan-Meier-Schätzer mit stratifizierten Log-Rank-, Wilcoxon- und
  Tarone-Ware-Tests. `PROC PHREG` passt das
  Cox-Proportional-Hazards-Modell über Breslow-Bindungen mithilfe
  des Python/statsmodels-Wrappers an. `PROC LOGISTIC` passt eine
  binär-logistische Maximum-Likelihood-Regression an.
- **Methoden der Risiko-Komposition:** Der zusammengesetzte
  Einfluss-Score aus Abschnitt 11 normalisiert degree, log-PageRank,
  Jurisdiktionsbreite und Amtsdauer-Jahre auf `[0, 1]` und summiert
  mit festen Gewichten (30/30/20/20). Der zusammengesetzte
  Entitäts-Risiko-Score aus Abschnitt 15 normalisiert die gedeckelte
  Amtsträgeranzahl, log-PageRank, das CTHI-Risikogewicht und ein
  binäres Kennzeichen für sanktionierte Amtsträger und summiert mit
  gleichen Gewichten von je 0,25.

Die vollständige Analyse ist aus dem Smoke-Test-Skript in diesem
Ordner reproduzierbar: `./smoke.jenner`. Es durchgängig gegen den
gemeinsamen `step-neo4j`-Dienst auszuführen (mit gesetztem
`JENNER_NEO4J_HOST` und `JENNER_NEO4J_PASS`, was die Plattform in
jedem Workspace-Pod für Sie erledigt) dauert rund zwei Minuten und
verifiziert, dass jede Abfrage und jede PROC — einschließlich der
fünf neuen Abschnitte, die neben der bestehenden Pipeline
hinzugefügt wurden — auf dem echten Graphen mit 163.435 Knoten die
erwartete Ausgabe liefert.
